# Day 07: Flow Matching in 1D

Welcome to practical 7! Learn a time-dependent vector field that transports noise to data (inspired by MIT Lab Two).

This notebook follows the MIT lab style: abstract base classes, PyTorch, and LaTeX in markdown. Fill in methods marked `# YOUR CODE HERE`.

In [ ]:
import math
from abc import ABC, abstractmethod
from typing import Optional

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

device = torch.device("cpu")
torch.manual_seed(42)
np.random.seed(42)

Conditional flow matching uses paths $x_t = (1-t) x_0 + t x_1$ with target velocity $u_t(x_t|x_1) = x_1 - x_0$. We regress a network $v_\theta(t, x_t)$ onto this target.

In [ ]:
class Sampleable(ABC):
    @abstractmethod
    def sample(self, n: int) -> torch.Tensor:
        pass

class GaussianMixture1D(Sampleable):
    def __init__(self, means, stds, weights):
        self.means = torch.tensor(means, dtype=torch.float32)
        self.stds = torch.tensor(stds, dtype=torch.float32)
        self.weights = torch.tensor(weights, dtype=torch.float32)

    def sample(self, n: int) -> torch.Tensor:
        idx = torch.multinomial(self.weights, n, replacement=True)
        return self.means[idx] + self.stds[idx] * torch.randn(n)

data_dist = GaussianMixture1D([[-2.0], [2.0]], [0.3, 0.3], [0.5, 0.5])
noise_dist = GaussianMixture1D([[0.0]], [1.0], [1.0])

### Exercise 7.1: Conditional path

**Your job**: Sample $(x_0, x_1, t)$ and compute $x_t$ and target velocity.

In [ ]:
def sample_conditional_path(n, t=None):
    # YOUR CODE HERE
    x0 = noise_dist.sample(n).unsqueeze(1)
    x1 = data_dist.sample(n).unsqueeze(1)
    if t is None:
        t = torch.rand(n, 1)
    xt = (1 - t) * x0 + t * x1
    velocity = x1 - x0
    return x0, x1, t, xt, velocity

x0, x1, t, xt, v = sample_conditional_path(8)
print("xt shape:", xt.shape, "velocity shape:", v.shape)

### Exercise 7.2: Velocity network

**Your job**: MLP taking concatenated $(t, x_t)$.

In [ ]:
class VelocityNet(nn.Module):
    def __init__(self):
        super().__init__()
        # YOUR CODE HERE
        self.net = nn.Sequential(
            nn.Linear(2, 64), nn.SiLU(),
            nn.Linear(64, 64), nn.SiLU(),
            nn.Linear(64, 1),
        )

    def forward(self, t, xt):
        inp = torch.cat([t, xt], dim=-1)
        return self.net(inp)

vel_net = VelocityNet().to(device)
opt = torch.optim.Adam(vel_net.parameters(), lr=1e-3)

### Exercise 7.3: Train flow matching

**Your job**: Minimize MSE between predicted and target velocity.

In [ ]:
for step in range(300):
    vel_net.train()
    opt.zero_grad()
    _, _, t, xt, target_v = sample_conditional_path(256)
    t, xt, target_v = t.to(device), xt.to(device), target_v.to(device)
    # YOUR CODE HERE
    pred_v = vel_net(t, xt)
    loss = F.mse_loss(pred_v, target_v)
    loss.backward()
    opt.step()
    if (step + 1) % 100 == 0:
        print(f"Step {step+1}: loss={loss.item():.4f}")

### Exercise 7.4: Integrate ODE from noise

**Your job**: Euler steps from $t=0$ to $t=1$.

In [ ]:
@torch.no_grad()
def integrate_flow(model, x_init, steps=50):
    x = x_init.clone()
    dt = 1.0 / steps
    for i in range(steps):
        t = torch.full((x.size(0), 1), i * dt)
        # YOUR CODE HERE
        x = x + dt * model(t, x)
    return x

x_init = noise_dist.sample(1000).unsqueeze(1)
generated = integrate_flow(vel_net, x_init).squeeze().numpy()
data_samples = data_dist.sample(1000).numpy()

plt.hist(data_samples, bins=40, alpha=0.5, label="data", density=True)
plt.hist(generated, bins=40, alpha=0.5, label="flow samples", density=True)
plt.legend()
plt.title("1D flow matching")
plt.show()

## Reflection (Day 7)

Take a few minutes to answer in your own words:

1. What was the most important concept you practiced today (flow matching)?
2. Where did you get stuck, and how did you resolve it?
3. How would you explain today's material to a classmate in two sentences?
4. What would you like to explore further?